<a href="https://colab.research.google.com/github/JuanMauwu/udea-ai4eng-20252-saberpro/blob/main/07%20-%20modelo%20ensemble%20voting%20rapido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROYECTO KAGGLE: Modelo adicional - ensemble voting rapido**
---

## **Descarga y preparación del conjunto de datos**

In [ ]:
import os

os.environ['KAGGLE_CONFIG_DIR'] = '.'
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 367MB/s]
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  inflating: submission_example.csv  
  inflating: test.csv                
  inflating: train.csv               


## **Modelo completo**

In [ ]:
#Importaciones

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.ensemble import VotingClassifier
import xgboost as xgb
import lightgbm as lgb
import joblib


#Carga de datos

print(">>> Cargando datasets")
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")
test_ids = df_test['ID']


#Preprocesamiento

def preprocesar_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    #Eliminar columnas irrelevantes
    cols_to_drop = ['periodo_academico', 'id']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    #Limpieza de texto
    df = df.map(lambda x: str(x).strip().lower() if isinstance(x, str) else x)
    df = df.replace({'sí': 'si', 'n': 'no'})

    #Mapeos Ordinales
    mapeos = {
        'e_valormatriculauniversidad': {
            'menos de 2.5 millones': 1, 'entre 2.5 millones y menos de 4 millones': 2,
            'entre 4 millones y menos de 5.5 millones': 3, 'entre 5.5 millones y menos de 7 millones': 4,
            'más de 7 millones': 5
        },
        'e_horassemanatrabaja': {
            '0': 0, 'menos de 10 horas': 1, 'entre 11 y 20 horas': 2,
            'entre 21 y 30 horas': 3, 'más de 30 horas': 4
        },
        'f_estratovivienda': {
            'estrato 1': 1, 'estrato 2': 2, 'estrato 3': 3,
            'estrato 4': 4, 'estrato 5': 5, 'estrato 6': 6
        },
        'f_educacionpadre': {
            'ninguno': 0, 'primaria completa': 1, 'secundaria (bachillerato) completa': 2,
            'técnica o tecnológica incompleta': 3, 'técnica o tecnológica completa': 4,
            'universitario': 5, 'postgrado': 6, 'no sabe': 0, 'desconocido': 0
        },
        'f_educacionmadre': {
            'ninguno': 0, 'primaria completa': 1, 'secundaria (bachillerato) completa': 2,
            'técnica o tecnológica incompleta': 3, 'técnica o tecnológica completa': 4,
            'universitario': 5, 'postgrado': 6, 'no sabe': 0, 'desconocido': 0
        }
    }

    for col, mapa in mapeos.items():
        if col in df.columns:
            df[col] = df[col].map(mapa)

    #Imputación Numérica
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in num_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    #Imputación Categórica y Binaria
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if col != 'rendimiento_global':
            if df[col].dropna().isin(['si', 'no']).any():
                df[col] = df[col].fillna('no').map({'si': 1, 'no': 0})
            else:
                df[col] = df[col].fillna('desconocido')

    return df

print(">>> Ejecutando preprocesamiento...")
X_train_clean = preprocesar_data(df_train)
X_test_clean = preprocesar_data(df_test)

#Separar Target (Temporal)
y_full = X_train_clean['rendimiento_global']
X_full = X_train_clean.drop(columns=['rendimiento_global'])

#Encode Target (Obligatorio para XGB/LGBM)
le = LabelEncoder()
y_encoded_full = le.fit_transform(y_full)
print(f"target classes: {le.classes_}")



#Reduccion del muestreo por memomria
print(">>> Aplicando muestreo para reducir el uso de RAM")
# Definimos el tamaño del muestreo (50% de los datos)
sample_fraction = 0.5

#Aplicar muestreo al dataset de entrenamiento
if sample_fraction < 1.0:

    X_sampled, _, y_sampled, _ = train_test_split(
        X_full,
        y_full,
        train_size=sample_fraction,
        random_state=42,
        stratify=y_full
    )

    #Actualizar X y y con la muestra
    X = X_sampled
    y = y_sampled
    y_encoded = le.transform(y)

    print(f"Dataset de entrenamiento reducido a {len(X)} filas (aprox. {sample_fraction*100:.0f}%)")

else:
    X = X_full
    y_encoded = y_encoded_full


#One-Hot Encoding y Alineación
print(">>> Aplicando One-Hot Encoding")
X = pd.get_dummies(X, columns=['e_prgm_academico', 'e_prgm_departamento'], drop_first=True)
X_test_final = pd.get_dummies(X_test_clean, columns=['e_prgm_academico', 'e_prgm_departamento'], drop_first=True)

#Alineación estricta de columnas (rellenar faltantes con 0)
X, X_test_final = X.align(X_test_final, join='left', axis=1, fill_value=0)

#Escalado
print("Escalando variables numéricas")
scaler = StandardScaler()
scale_cols = ['indicador_1', 'indicador_2', 'indicador_3', 'indicador_4']
#Validar que existan
scale_cols = [c for c in scale_cols if c in X.columns]
X[scale_cols] = scaler.fit_transform(X[scale_cols])
X_test_final[scale_cols] = scaler.transform(X_test_final[scale_cols])

print(f"Dimensiones finales: Train {X.shape}, Test {X_test_final.shape}")


#Ajuste para LIGHTGBM/XGBOOST (Limpieza de nombres de columnas)
print(">>> Limpiando nombres de columnas (Fix LightGBM)...")
X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
X_test_final.columns = X_test_final.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)



#MODELO


print(">>> Configurando Ensemble (XGBoost + LightGBM)")

#Modelo 1: XGBoost (Configuración RÁPIDA y balanceada)
clf_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='merror'
)

#Modelo 2: LightGBM (Configuración RÁPIDA y balanceada)
clf_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

#Ensemble: Voting Classifier (Soft Voting promedia las probabilidades)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', clf_xgb),
        ('lgb', clf_lgb)
    ],
    voting='soft'
)

#Se omite la validación cruzada
print(">>> Entrenamiento directo del modelo final sin Validación Cruzada (para ahorrar recursos).")

#Entrenamiento Final
print(">>> Entrenando modelo final con los datos muestreados")
# Pasamos X.values y y_encoded de la muestra
voting_clf.fit(X.values, y_encoded)


#Generación de submission
print(">>> Prediciendo test set...")
# Pasamos X_test_final.values
preds_encoded = voting_clf.predict(X_test_final.values)
preds_final = le.inverse_transform(preds_encoded)

submission = pd.DataFrame({
    'ID': test_ids,
    'RENDIMIENTO_GLOBAL': preds_final
})

#Archivo para kaggle
output_file = "submission_ensemble_rapido_final.csv"
submission.to_csv(output_file, index=False)
print(f">>> ¡Éxito! Archivo guardado como: {output_file}")

>>> Cargando datasets...
>>> Ejecutando preprocesamiento...
target classes: ['alto' 'bajo' 'medio-alto' 'medio-bajo']
>>> Aplicando muestreo para reducir el uso de RAM...
Dataset de entrenamiento reducido a 346250 filas (aprox. 50%)
>>> Aplicando One-Hot Encoding...
Escalando variables numéricas...
Dimensiones finales: Train (346250, 970), Test (296786, 970)
>>> Limpiando nombres de columnas (Fix LightGBM)...
>>> Configurando Ensemble (XGBoost + LightGBM)...
>>> Entrenamiento directo del modelo final sin Validación Cruzada (para ahorrar recursos).
>>> Entrenando modelo final con los datos muestreados...
>>> Prediciendo test set...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


>>> ¡Éxito! Archivo guardado como: submission_ensemble_rapido_final.csv
